# Create from Document — Colab (GPU)

Upload a document (topic/writing) and get a **narrated infographic video** —
the AI designs the infographic, clones a voice to narrate it, and builds the
video with subtitles. Runs on Colab's GPU (L4 recommended).

**First:** `Runtime -> Change runtime type -> GPU (L4) -> Save`. Then run the
cells top to bottom (Shift+Enter).

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU: Runtime -> Change runtime type -> GPU, then re-run.'

## 2. Download the project (public repo, no token)

In [ ]:
import os
USER='PrashantRMarathe'; REPO='voice-replacer'
if os.path.isdir(REPO):
    # Always pull the latest fixes automatically (tiny download).
    !cd {REPO} && git pull --quiet
else:
    !git clone https://github.com/{USER}/{REPO}.git
%cd {REPO}
print('Now in:', os.getcwd(), '(latest version)')

## 3. Install dependencies (~5 min, first run)

In [ ]:
# Voice + transcription + translation (the essentials)
!pip install -q coqui-tts 'transformers<5' ffmpeg-python deep-translator num2words soundfile openai-whisper
# F5-TTS (default voice engine — softest for English)
!pip install -q f5-tts 2>/dev/null || echo "NOTE: f5-tts not installed; will use XTTS."
# Mode 2: document readers + HTML->image rendering (essentials)
!pip install -q python-docx pypdf playwright
!playwright install chromium
!playwright install-deps chromium 2>/dev/null || true
print('Dependencies installed (core).')

## 4. Configure (GPU quality settings)

On the GPU we turn ON the real AI brain and the Ken Burns motion.

In [ ]:
import os, importlib
os.environ['COQUI_TOS_AGREED'] = '1'
import config
config.ENABLE_LLM_BRAIN = True          # AI designs the infographic + script
config.LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'  # small+fast (avoids long waits)
config.KEN_BURNS = True                 # slow zoom motion (fine on GPU)
config.WHISPER_BACKEND = 'openai'       # subtitle timing (avoids PyAV issue)
config.WHISPER_MODEL = 'small'          # subtitles only; small is plenty

# ---- Voice: same soft F5 setup as dubbing (this is what fixed the voice) ----
config.TTS_BACKEND = 'f5'
config.ENABLE_VOICE_ENHANCE = False     # no denoiser hardening the voice
config.ENABLE_AUDIO_CLEANUP = False     # no loudness compression (keeps softness)

# Guarantee F5 is active; auto-install if missing.
import tts
if tts.active_backend_name() != 'f5':
    print('F5 not ready - installing automatically...')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'f5-tts'])
    importlib.invalidate_caches()
    tts._backend = None

print('Device:', config.DEVICE, '| Voice engine:', tts.active_backend_name().upper(),
      '| LLM:', config.LLM_MODEL)

## 5. Upload your document and a narration voice

- **Document:** .txt / .md / .docx / .pdf with your topic/writing.
- **Voice:** a clean 15-30s clip (MP3/WAV) to narrate in.

In [ ]:
from google.colab import files
print('Select your DOCUMENT...')
doc_path = list(files.upload().keys())[0]
print('Select your VOICE sample...')
ref_path = list(files.upload().keys())[0]
print('Document:', doc_path, '| Voice:', ref_path)

## 6. Create the video

In [ ]:
import create
out = create.create_from_document(doc_path, reference_path=ref_path, cb=print)
print('\nDONE ->', out)

## 7. Download the video (+ subtitles)

In [ ]:
from google.colab import files
from pathlib import Path
files.download(out)
stem = Path(out).with_suffix('')
for ext in ('.srt', '.vtt'):
    p = f'{stem}{ext}'
    if Path(p).is_file():
        files.download(p)